# Realized Profit and Loss

This notebook reads the persisted realized profit/loss records generated from transactions. Sales use the weighted acquisition cost from the position at the time of sale; income, dividends, and interest use the transaction operation value.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from config import Config
from src.core.repositories.asset_repository import AssetRepository
from src.core.repositories.profit_loss_repository import ProfitLossRepository
from src.core.repositories.transaction_repository import TransactionRepository

In [ ]:
config = Config()
profit_loss_repository = ProfitLossRepository(config=config)
transaction_repository = TransactionRepository(config=config)
asset_repository = AssetRepository(config=config)

profit_losses = profit_loss_repository.all()
transactions = {transaction.transaction_id: transaction for transaction in transaction_repository.all()}
assets = {asset.asset_id: asset for asset in asset_repository.all()}

print(f'Loaded {len(profit_losses)} persisted profit/loss records.')

In [ ]:
import pandas as pd
from IPython.display import display

profit_loss_table = pd.DataFrame([
    {
        'asset_id': item.asset_id,
        'asset_name': assets[item.asset_id].asset_name,
        'transaction_id': item.transaction_id,
        'holding_id': item.holding_id,
        'date': transactions[item.transaction_id].date,
        'operation_type': transactions[item.transaction_id].operation_type.value,
        'value': item.value,
        'type': item.type.value,
    }
    for item in profit_losses
]).sort_values(['date', 'transaction_id']).reset_index(drop=True)

display(profit_loss_table)

In [ ]:
summary_by_asset = (
    profit_loss_table.groupby(['asset_id', 'asset_name'], as_index=False)['value']
    .sum()
    .sort_values('value', ascending=False)
)
display(summary_by_asset)

print(f'Total realized profit/loss: {profit_loss_table["value"].sum()}')

In [ ]:
monthly_profit_loss = profit_loss_table.assign(
    month=pd.to_datetime(profit_loss_table['date']).dt.to_period('M').astype(str)
).groupby('month', as_index=False)['value'].sum()

display(monthly_profit_loss)
print(f'Months with realized profit/loss: {len(monthly_profit_loss)}')

In [ ]:
import matplotlib.pyplot as plt

months = monthly_profit_loss['month'].tolist()
values = [float(value) for value in monthly_profit_loss['value']]
colors = ['#2a9d8f' if value >= 0 else '#e76f51' for value in values]

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(months, values, color=colors)
ax.axhline(0, color='#333333', linewidth=0.8)
ax.set_title('Monthly Realized Profit/Loss')
ax.set_xlabel('Month')
ax.set_ylabel('Amount')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
assert all(item.type.value == 'realized' for item in profit_losses)
print('Verified that all persisted records are realized.')